In [1]:
#Written by RIG on 12/9/19
#Picked back up on 1/5/20

#This script automates pre-processing of 2p imaging data files/folders

#Import the necessary modules
import os, sys
import shutil
import glob
import h5py
import numpy as np
from PIL import Image
from skimage import io
#import tempfile

In [ ]:
#######Define parent directory########
#parent_dir = r"E:\Rachel\Test\230407_PrL-NAc-G6-8M_HI-D2_FOV2"
parent_dir = input(r'E:\Ian_Pav_Stim\240827_CARL10_PC-D25_FOV9\CARL10_PC-D25_FOV9_behavior-002')
os.chdir(parent_dir)
######################################

print ("The current working directory is %s" % parent_dir)

#Make extras folders
folders = next(os.walk(parent_dir))[1]
extras_path = os.path.join(parent_dir,"extras")
if "extras" not in folders:
    os.mkdir(extras_path)
    for i in folders:
        os.mkdir(os.path.join(extras_path,i))
    print ("Extras folder created in %s" % parent_dir)
else:
    print ("Extras folder already in %s" % parent_dir)

In [3]:
#Find, move, and rename "References" folders, .env and .xml files
for folder in next(os.walk(parent_dir))[1]:
    if "extras" not in folder:
        if next(os.walk(os.path.join(parent_dir,folder)))[1] == ['References']:
            ref_path = os.path.join(parent_dir,folder,'References')
            shutil.move(ref_path, os.path.join(extras_path,folder))
            print("References moved to %s" %extras_path, folder)
        for datafile in next(os.walk(os.path.join(parent_dir,folder)))[2]:
            if ".env" in datafile:
                env_path = os.path.join(parent_dir,folder,datafile)
                shutil.move(env_path, os.path.join(extras_path,folder,datafile))
                print(".envs moved to %s" %extras_path, folder)
            elif ".xml" in datafile:
                xml_path = os.path.join(parent_dir,folder,datafile)
                shutil.move(xml_path, os.path.join(extras_path,folder,datafile))
                print(".xmls moved to %s" %extras_path, folder)

('References moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2-000')
('.envs moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2-000')
('.xmls moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2-000')
('References moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2_beh-003')
('.envs moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2_beh-003')
('.xmls moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2_beh-003')
('References moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-NAc-CARL-2M_HIGH-D2\\extras', '231206_PrL-NAc-CARL-2M_HIGH-D2_FOV2_POST-004')
('.envs moved to E:\\Rachel\\PrL-NAc-CARL-2M\\231206_PrL-N

In [ ]:
#Remove .ome from TIF filenames
for folder in next(os.walk(parent_dir))[1]:
    for datafile in next(os.walk(os.path.join(parent_dir,folder)))[2]:
        if "ome" in datafile:
            tif_path = os.path.join(parent_dir,folder,datafile)
            tif_path,ext = os.path.splitext(tif_path)
            new_tif_path = tif_path[:-4]
            tif_path = os.path.join(tif_path+ext)
            new_tif_path = os.path.join(new_tif_path+ext)
            os.rename(tif_path, new_tif_path)

In [2]:
#Written by Vijay Namboodiri of Stuber Lab. Updated for python 3.x by Mike Martino in the Otis lab (2023).

#This script takes in a source directory as argument and finds all the input directories within that directory.
#-- For each such input directory, it finds all the .tif files in that (input) directory, sorts them according to the
#   trailing number in filenames and combines them into a single hdf5 file with the same name as the input directory name
#   and stores this file in the respective input directory. If the tif files don't contain a trailing number, then the
#   script will throw an error.
#-- If there are no such input directories, it treats the source directory as the input directory and looks for tif files
#   directly within the source directory and converts and combines them.
#-- The tif files can contain any number of frames. The number of frames per file can be different across combined files.
#
#Usage: Go to terminal (on Windows, run cmd.exe) and run the following command
#
#       python /path/to/write_hdf5_from_tif.py path/to/source_directory --compression compressionamount
#
#compressionamount is specified as a number between 1 (lowest) and 9 (highest)).
#If --compression is not specified, default = None, i.e. no compression.
#So don't specify anything, if you don't want any compression.
#
#Note: On Windows, the path slashes are backslashes and not forward slashes. Also, if your current directory is the one
#      containing this file, you don't have to specify the path.

#"""
from PIL import Image
import numpy as np
import os
import h5py
import re
import itertools as it
from argparse import ArgumentParser
from multiprocessing import Pool, cpu_count

def calculate_num_frames(tiffile):
    num_frames = 0
    f = Image.open(tiffile, 'r')
    try:
        while True:
            f.seek(num_frames)
            num_frames = num_frames + 1
    except EOFError:
        pass

    return num_frames, f


def write_into_hdf5(input_filenames, output_filename, group='/', key='imaging', compression=None):
    name = os.path.join(group, key)
    file_found = False
    for frame in input_filenames:
        try:
            f = Image.open(frame, 'r')
        except IOError:
            pass
        else:
            f.seek(0)
            first_img = np.array(f)
            file_found = True
            break

    if not file_found:
        raise IOError(('No good files found: {}').format(output_filename))
    total_num_frames = 0
    for tiffile in input_filenames:
        num_frames, f = calculate_num_frames(tiffile)
        total_num_frames += num_frames

    output_shape = (
     total_num_frames, first_img.shape[0], first_img.shape[1])
    h5 = h5py.File(output_filename, 'w', libver='latest')
    h5.create_dataset(name, output_shape, dtype=first_img.dtype, maxshape=output_shape, chunks=(
    1, output_shape[1], output_shape[2]), compression=compression if compression else None)

    try:
        num_frames_till_now = 0
        for file_idx, tiffile in zip(it.count(), input_filenames):
            num_frames, f = calculate_num_frames(tiffile)
            for tempframe_idx in range(num_frames):
                f.seek(tempframe_idx)
                f_data = np.array(f)
                frame_idx = num_frames_till_now + tempframe_idx
                h5[name][frame_idx, :, :] = f_data

            num_frames_till_now = num_frames_till_now + num_frames

        for idx, label in enumerate(['t', 'y', 'x']):
            h5[name].dims[idx].label = label

        h5.close()
    except:
        h5.close()
        os.remove(output_filename)
        raise
    else:
        try:
            h5 = h5py.File(output_filename, 'r')
            num_frames_till_now = 0
            for file_idx, tiffile in zip(it.count(), input_filenames):
                num_frames, f = calculate_num_frames(tiffile)
                for tempframe_idx in range(num_frames):
                    frame_idx = num_frames_till_now + tempframe_idx
                    f.seek(tempframe_idx)
                    if not np.all(h5[name][frame_idx, :, :] == np.array(f)):
                        raise AssertionError

                num_frames_till_now = num_frames_till_now + num_frames
                f.close()

            h5.close()
        except:
            h5.close()
            os.remove(output_filename)
            raise


def execute_single_dir(args):
    input_dir, output_file, compression = args
    os.chdir(input_dir)
    output_filename = os.path.join(os.path.dirname(input_dir), os.path.basename(input_dir) + '.h5')
    contents = os.listdir(input_dir)
    filtered = [c for c in contents if c[-3:] == 'tif']
    try:
        input_filenames = sorted(filtered, key=lambda filename: int(re.search('\\d+$', os.path.splitext(filename)[0]).group()))
    except AttributeError:
        raise Exception('One filename does not have a trailing number. Please add a trailing number and re-run')

    write_into_hdf5(input_filenames, output_filename, compression=compression)

def main():
    source_dir = input("Path to tif folder")
    output_file = input("Path to output file")
    compression = None
    data_dirs = []
    for dirpath, dirnames, filenames in os.walk(source_dir):
        if any(filename.endswith('.tif') for filename in filenames):
            data_dirs.append(dirpath)
    
    if data_dirs:
        execute_single_dir((dirpath, output_file, compression))
    else:
        print("No tif files found in " + source_dir)

if __name__ == '__main__':
    main()

Path to tif folderr'E:\Ian Pav Stim\240829_CARL10_PC-D26_FOV9_C1-STIM\CARL10_PC-D26_FOV9_C1-STIM_3cells_behavior-011'
Path to output filer'E:\Ian Pav Stim\240829_CARL10_PC-D26_FOV9_C1-STIM'
